In [1]:
import pandas as pd
from rapidfuzz import process, fuzz
import unidecode
import re

In [2]:
categories = pd.read_parquet(r'../export/small_parquet/municiplity.parquet')
categories

,DE_MUNICIP
0,ABRERA
1,ACEBEDO
2,ADEJE CASCO
3,AGUILON
4,ALAMEDA DE LA SAGRA
...,...
804,VILLASILA DE VALDAVIA
805,VILLAZANZO DE VALDERADUEY
806,VITORIA-GASTEIZ
807,VIVEIRO


In [3]:
categories.nunique()

DE_MUNICIP    809
dtype: int64

In [4]:
categories.dtypes

DE_MUNICIP    object
dtype: object

In [5]:
categories['DE_MUNICIP'] = categories['DE_MUNICIP'].str.strip()

In [6]:
categories.sort_values(by='DE_MUNICIP', inplace=True)
categories.head()

,DE_MUNICIP
201,
202,A CORUÃ‘A
203,ABIA DE LAS TORRES
0,ABRERA
1,ACEBEDO


In [7]:
categories.isna().sum()

DE_MUNICIP    0
dtype: int64

In [8]:
categories = categories[categories['DE_MUNICIP'].str.strip() != '']
categories.shape

(808, 1)

In [9]:
latlong_data = pd.read_csv(r"..\BD_MUNICIPIOS-ENTIDADES\MUNICIPIOS.csv",
                           delimiter=';',
                           encoding='latin1')
latlong_data.head()

,COD_INE,ID_REL,COD_GEO,COD_PROV,PROVINCIA,NOMBRE_ACTUAL,POBLACION_MUNI,SUPERFICIE,PERIMETRO,COD_INE_CAPITAL,CAPITAL,POBLACION_CAPITAL,HOJA_MTN25,LONGITUD_ETRS89_REGCAN95,LATITUD_ETRS89_REGCAN95,ORIGENCOOR,ALTITUD,ORIGENALTITUD
0,1001000000,1010014,1010,1,Araba/Álava,Alegría-Dulantzi,2961,"1994,5872",35069,1001000101,Alegría-Dulantzi,2842,0113-3,"-2,512507724","42,84045247",Detección automática,568,MDT
1,1002000000,1010029,1020,1,Araba/Álava,Amurrio,10346,"9617,86",65701,1002000201,Amurrio,9256,0086-4,"-3,001015194","43,05265767",Detección automática,217,MDT
2,1003000000,1010035,1030,1,Araba/Álava,Aramaio,1353,"7308,96",42097,1003000601,Ibarra,731,0087-4,"-2,564829379","43,05257873",Detección automática,325,MDT
3,1004000000,1010040,1040,1,Araba/Álava,Artziniega,1868,"2728,73",22886,1004000101,Artziniega,1732,0086-1,"-3,13052099","43,1217919",Detección automática,199,MDT
4,1006000000,1010066,1060,1,Araba/Álava,Armiñón,233,"1297,27",24707,1006000101,Armiñón,106,0137-4,"-2,872270813","42,72340924",Detección automática,466,MDT


In [10]:
latlong_data = latlong_data[['NOMBRE_ACTUAL', 'PROVINCIA', 'LONGITUD_ETRS89_REGCAN95', 'LATITUD_ETRS89_REGCAN95']]
latlong_data.shape

(8132, 4)

In [11]:
latlong_data.rename(columns={
    'NOMBRE_ACTUAL':'DE_MUNICIP',
    'PROVINCIA':'PROVINCE',
    'LONGITUD_ETRS89_REGCAN95':'LONGITUDE',
    'LATITUD_ETRS89_REGCAN95':'LATITUDE'
},
inplace=True)
latlong_data.head()

,DE_MUNICIP,PROVINCE,LONGITUDE,LATITUDE
0,Alegría-Dulantzi,Araba/Álava,"-2,512507724","42,84045247"
1,Amurrio,Araba/Álava,"-3,001015194","43,05265767"
2,Aramaio,Araba/Álava,"-2,564829379","43,05257873"
3,Artziniega,Araba/Álava,"-3,13052099","43,1217919"
4,Armiñón,Araba/Álava,"-2,872270813","42,72340924"


In [12]:
latlong_data.dtypes

DE_MUNICIP    object
PROVINCE      object
LONGITUDE     object
LATITUDE      object
dtype: object

In [13]:
latlong_data['DE_MUNICIP'] = latlong_data['DE_MUNICIP'].str.strip().str.upper()
latlong_data['PROVINCE'] = latlong_data['PROVINCE'].str.strip()
latlong_data['LATITUDE'] = latlong_data['LATITUDE'].str.replace(',', '.').astype('float64')
latlong_data['LONGITUDE'] = latlong_data['LONGITUDE'].str.replace(',', '.').astype('float64')
print(latlong_data.dtypes)
latlong_data.head()

DE_MUNICIP     object
PROVINCE       object
LONGITUDE     float64
LATITUDE      float64
dtype: object


,DE_MUNICIP,PROVINCE,LONGITUDE,LATITUDE
0,ALEGRÍA-DULANTZI,Araba/Álava,-2.512508,42.840452
1,AMURRIO,Araba/Álava,-3.001015,43.052658
2,ARAMAIO,Araba/Álava,-2.564829,43.052579
3,ARTZINIEGA,Araba/Álava,-3.130521,43.121792
4,ARMIÑÓN,Araba/Álava,-2.872271,42.723409


In [14]:
latlong_data.sort_values(by='DE_MUNICIP', inplace=True)
latlong_data.head()

,DE_MUNICIP,PROVINCE,LONGITUDE,LATITUDE
4891,A ARNOIA,Ourense,-8.135751,42.253066
2132,A BAÑA,A Coruña,-8.758003,42.961898
4902,A BOLA,Ourense,-7.928518,42.140937
2143,A CAPELA,A Coruña,-8.068806,43.435552
5292,A CAÑIZA,Pontevedra,-8.273359,42.212754


In [15]:
merged = categories.merge(latlong_data, on=['DE_MUNICIP'], how='left')
merged

,DE_MUNICIP,PROVINCE,LONGITUDE,LATITUDE
0,A CORUÃ‘A,NaN,NaN,NaN
1,ABIA DE LAS TORRES,Palencia,-4.421902,42.420220
2,ABRERA,Barcelona,1.901569,41.516398
3,ACEBEDO,León,-5.115871,43.040732
4,ACEUCHAL,Badajoz,-6.487251,38.647755
...,...,...,...,...
809,YUNQUERA,Málaga,-4.917332,36.734203
810,ZAFRA,Badajoz,-6.418559,38.426344
811,ZARAGOZA,Zaragoza,-0.877318,41.656208
812,ZARZOSA DE RIO PISUERGA,NaN,NaN,NaN


In [16]:
merged[merged['LONGITUDE'].isna()]

,DE_MUNICIP,PROVINCE,LONGITUDE,LATITUDE
0,A CORUÃ‘A,NaN,NaN,NaN
6,ADEJE CASCO,NaN,NaN,NaN
10,AGUILON,NaN,NaN,NaN
11,AGUIMES,NaN,NaN,NaN
12,AGÃœIMES,NaN,NaN,NaN
...,...,...,...,...
798,VILLAYUSO,NaN,NaN,NaN
800,VILOBI D'ONYAR,NaN,NaN,NaN
801,VINAROS,NaN,NaN,NaN
805,XUNQUEIRA DE AMBIA,NaN,NaN,NaN


In [17]:
merged[merged['DE_MUNICIP'].duplicated(keep=False)]

,DE_MUNICIP,PROVINCE,LONGITUDE,LATITUDE
131,CABANES,Castelló/Castellón,0.045412,40.156104
132,CABANES,Girona,2.977957,42.307504
199,CIEZA,Murcia,-1.427730,38.236594
200,CIEZA,Cantabria,-4.096722,43.221056
433,MIERES,Girona,2.640292,42.123296
434,MIERES,Asturias,-5.772657,43.248794
579,SADA,Navarra,-1.397668,42.585825
580,SADA,A Coruña,-8.253590,43.351531
718,TORRENT,València/Valencia,-0.465982,39.436721
719,TORRENT,Girona,3.128197,41.951843


In [18]:
categories_prov = pd.read_parquet(r'..\export\small_parquet\muncipility_province.parquet')
print(categories_prov.shape)
categories_prov.head()

(778, 2)


,Municipality,Province
0,Abla,Almería
1,Abrucena,Almería
2,Adamuz,Córdoba
3,Adra,Almería
4,Agrón,Granada


In [19]:
same_name_diff_prov = merged[merged['DE_MUNICIP'].duplicated()]['DE_MUNICIP'].values
same_name_diff_prov

array(['CABANES', 'CIEZA', 'MIERES', 'SADA', 'TORRENT', 'VILLAESCUSA'],
      dtype=object)

In [20]:
categories_prov[categories_prov['Municipality'].isin(same_name_diff_prov)]

,Municipality,Province


In [21]:
def clean_municipality(name):
    if not isinstance(name, str): return ""
    # 1. Fix specific encoding artifacts
    name = name.replace("Ã‘", "Ñ").replace("Ã“", "Ó")
    
    # 2. Expand common abbreviations
    name = name.replace("S/C", "SANTA CRUZ")
    name = name.replace("S.", "SAN")
    
    # 3. Fix trailing articles e.g., "CUERVO (EL)" -> "EL CUERVO"
    name = re.sub(r'^(.*?)\s*\((EL|LA|LOS|LAS)\)$', r'\2 \1', name)
    
    # 4. Remove accents and convert to uppercase for baseline comparison
    name = unidecode.unidecode(name).upper()
    return name.strip()

# Apply cleaning to both lists
clean_dataset = [clean_municipality(name) for name in categories.DE_MUNICIP.tolist()]
clean_all = [clean_municipality(name) for name in latlong_data['DE_MUNICIP'].tolist()]

matches = {}
for original_name, clean_name in zip(categories.DE_MUNICIP.tolist(), clean_dataset):
    
    # Handle bilingual names by splitting at '/'
    parts = clean_name.split('/')
    best_match = None
    best_score = 0
    
    for part in parts:
        result = process.extractOne(
            part,
            clean_all,
            scorer=fuzz.token_sort_ratio,
            score_cutoff=85  # Raised significantly to prevent false positives
        )
        if result and result[1] > best_score:
            best_match = result
            best_score = result[1]
            
    if best_match:
        # Retrieve the original uncleaned name from the target dataframe using the index
        target_original_name = latlong_data['DE_MUNICIP'].iloc[best_match[2]]
        matches[original_name] = target_original_name
    else:
        matches[original_name] = None

In [22]:
matches

{'A CORUÃ‘A': 'A CORUÑA',
 'ABIA DE LAS TORRES': 'ABIA DE LAS TORRES',
 'ABRERA': 'ABRERA',
 'ACEBEDO': 'ACEBEDO',
 'ACEUCHAL': 'ACEUCHAL',
 'ADEJE': 'ADEJE',
 'ADEJE CASCO': None,
 'ADEMUZ': 'ADEMUZ',
 'AGRAMUNT': 'AGRAMUNT',
 'AGUADULCE': 'AGUADULCE',
 'AGUILON': 'AGUILÓN',
 'AGUIMES': 'AGÜIMES',
 'AGÃœIMES': None,
 'AITONA': 'AITONA',
 'AJO': None,
 'ALAMEDA DE LA SAGRA': 'ALAMEDA DE LA SAGRA',
 'ALARO': 'ALARÓ',
 'ALBACETE': 'ALBACETE',
 'ALBENDIEGO': 'ALBENDIEGO',
 'ALBOCASSER': 'ALBOCÀSSER',
 'ALBOLOTE': 'ALBOLOTE',
 'ALBURQUERQUE': 'ALBURQUERQUE',
 'ALCALA DE HENARES': 'ALCALÁ DE HENARES',
 'ALCALA DE XIVERT': 'ALCALÀ DE XIVERT',
 'ALCALA LA REAL': 'ALCALÁ LA REAL',
 'ALCANAR': 'ALCANAR',
 'ALCAUDETE': 'ALCAUDETE',
 'ALCAZAR DE SAN JUAN': 'ALCÁZAR DE SAN JUAN',
 'ALCAÃ‘IZ': 'ALCAÑIZ',
 'ALCOBENDAS': 'ALCOBENDAS',
 'ALCORCON': 'ALCORCÓN',
 'ALCUBIERRE': 'ALCUBIERRE',
 'ALCUDIA': 'ALCÚDIA',
 'ALDEASECA DE ALBA': 'ALDEASECA DE ALBA',
 'ALDEATEJADA': 'ALDEATEJADA',
 'ALFORJA': 'ALFO

In [23]:
manual_corrections = {
    'ADEJE CASCO': 'ADEJE',
    'AGÃœIMES': 'AGÜIMES',
    'AJO': 'BAREYO', 
    'ALICANTE/ALACANT': 'ALACANT/ALICANTE',
    'ALMAZORA': 'ALMASSORA',
    'ALQUERIAS DEL NIÃ‘O PERDIDO': 'LES ALQUERIES/ALQUERÍAS DEL NIÑO PERDIDO',
    'BARRIO DE BRICIA': 'ALFOZ DE BRICIA',
    'BENICASIM': 'BENICÀSSIM/BENICASIM',
    'BORRIANA/BURRIANA': 'BORRIANA/BURRIANA',
    'BURRIANA': 'BORRIANA/BURRIANA',
    'CABANAQUINTA/CABAÃ‘AQUINTA': 'ALLER', 
    'CASTELLON DE LA PLANA': 'CASTELLÓ DE LA PLANA/CASTELLÓN DE LA PLANA',
    'EJEA': 'EJEA DE LOS CABALLEROS',
    'EL GOLFO': 'YAIZA', 
    'ELCHE': 'ELX/ELCHE',
    'ELCHE/ELX': 'ELX/ELCHE',
    'ELEXALDE': 'GALDAKAO', 
    'JAVEA': 'XÀBIA/JÁVEA',
    'JAVEA/XABIA': 'XÀBIA/JÁVEA',
    'LA ALMUNIA': 'LA ALMUNIA DE DOÑA GODINA',
    'LA CONCHA': 'VILLAESCUSA', 
    'LA IGLESIA': 'RUILOBA', 
    'LA POLA': 'LENA', 
    'LAS PALMAS DE G.C.': 'LAS PALMAS DE GRAN CANARIA',
    'LLANSA': 'LLANÇÀ',
    'MATAMOROSA': 'CAMPOO DE ENMEDIO',
    'MIERES DEL CAMIN': 'MIERES',
    'MORON FRONTERA': 'MORÓN DE LA FRONTERA',
    'MURIEDAS': 'CAMARGO',
    'OSORNO': 'OSORNO LA MAYOR',
    'PALMA DE MALLORCA': 'PALMA', 
    'PAMPLONA': 'PAMPLONA/IRUÑA',
    'PEÃ‘ISCOLA': 'PENÍSCOLA/PEÑÍSCOLA',
    'POLLENÃ‡A': 'POLLENÇA',
    'POMALUENGO': 'CASTAÑEDA',
    'PUENTENANSA': 'RIONANSA',
    'RENEDO': 'PIÉLAGOS', 
    'RIVERO': 'SAN FELICES DE BUELNA',
    'RONCAL': 'RONCAL/ERRONKARI',
    'RUBAYO': 'MARINA DE CUDEYO',
    'SAN ILDEFONSO': 'REAL SITIO DE SAN ILDEFONSO',
    'SAN ILDEFONSO O LA GRANJA': 'REAL SITIO DE SAN ILDEFONSO',
    'SAN JOSE': 'NÍJAR', 
    'SAN MIGUEL DE LUENA': 'LUENA',
    'SAN MIGUEL DE MERUELO': 'MERUELO',
    'SAN VICENTE DE TORANZO': 'CORVERA DE TORANZO',
    'SANT CARLES DE LA RAPITA': 'LA RÀPITA', 
    'SETLA': 'ELS POBLETS', 
    'SIGÃœENZA': 'SIGÜENZA',
    'TAMA': 'CILLORIGO DE LIÉBANA', 
    'TARRIO': 'AMES', # Tarrio is ambiguous. It is a village not an official administrative municipio
    'TORRE BAJA': 'TORREBAJA',
    'TORRE ENDOMENECH': "LA TORRE D'EN DOMÉNEC",
    'UZTARROZ': 'UZTÁRROZ/UZTARROTZE',
    'VALDECILLA': 'MEDIO CUDEYO',
    'VEGUILLA': 'SOBA',
    'VILLAJOYOSA/LA VILA JOIOSA': 'LA VILA JOIOSA/VILLAJOYOSA',
    'VILLAYUSO': 'CIEZA'
}

In [24]:
final_corrections = matches | manual_corrections
final = pd.DataFrame.from_dict(final_corrections, orient='index').reset_index()
final.columns = ['DE_MUNICIP_org', 'DE_MUNICIP']
final

,DE_MUNICIP_org,DE_MUNICIP
0,A CORUÃ‘A,A CORUÑA
1,ABIA DE LAS TORRES,ABIA DE LAS TORRES
2,ABRERA,ABRERA
3,ACEBEDO,ACEBEDO
4,ACEUCHAL,ACEUCHAL
...,...,...
803,YUNQUERA,YUNQUERA
804,ZAFRA,ZAFRA
805,ZARAGOZA,ZARAGOZA
806,ZARZOSA DE RIO PISUERGA,ZARZOSA DE RÍO PISUERGA


In [25]:
final_merged = final.merge(latlong_data, on='DE_MUNICIP', how='left')
final_merged

,DE_MUNICIP_org,DE_MUNICIP,PROVINCE,LONGITUDE,LATITUDE
0,A CORUÃ‘A,A CORUÑA,A Coruña,-8.395826,43.371495
1,ABIA DE LAS TORRES,ABIA DE LAS TORRES,Palencia,-4.421902,42.420220
2,ABRERA,ABRERA,Barcelona,1.901569,41.516398
3,ACEBEDO,ACEBEDO,León,-5.115871,43.040732
4,ACEUCHAL,ACEUCHAL,Badajoz,-6.487251,38.647755
...,...,...,...,...,...
812,YUNQUERA,YUNQUERA,Málaga,-4.917332,36.734203
813,ZAFRA,ZAFRA,Badajoz,-6.418559,38.426344
814,ZARAGOZA,ZARAGOZA,Zaragoza,-0.877318,41.656208
815,ZARZOSA DE RIO PISUERGA,ZARZOSA DE RÍO PISUERGA,Burgos,-4.267593,42.536190


In [26]:
final_merged.isna().sum()

DE_MUNICIP_org    0
DE_MUNICIP        0
PROVINCE          1
LONGITUDE         1
LATITUDE          1
dtype: int64

In [27]:
final_merged[final_merged['LATITUDE'].isna()]

,DE_MUNICIP_org,DE_MUNICIP,PROVINCE,LONGITUDE,LATITUDE
734,UZTARROZ,UZTÁRROZ/UZTARROTZE,NaN,NaN,NaN


In [28]:
all_districts = latlong_data['DE_MUNICIP'].tolist()
process.extract('UZTARROZ', all_districts, limit=5, scorer=fuzz.token_sort_ratio)

[('URROZ', 76.92307692307692, 7147),
 ('MAZARAMBROZ', 63.1578947368421, 4383),
 ('BULARROS', 62.5, 1340),
 ('ZARAGOZA', 62.5, 8061),
 ('BARRO', 61.53846153846154, 939)]

Distinct Municipalities: Uztárroz (Uztarrotze) and Urroz are two completely different, separate towns located in the province of Navarre.

In [29]:
final_merged.loc[734, 'LATITUDE'] = 42.9194
final_merged.loc[734, 'LONGITUDE'] = -0.9567
final_merged.loc[734, 'PROVINCE'] = 'Navarre'

In [30]:
final_merged.isna().sum()

DE_MUNICIP_org    0
DE_MUNICIP        0
PROVINCE          0
LONGITUDE         0
LATITUDE          0
dtype: int64

In [31]:
final_merged[final_merged['DE_MUNICIP_org'].duplicated(keep=False)]

,DE_MUNICIP_org,DE_MUNICIP,PROVINCE,LONGITUDE,LATITUDE
131,CABANES,CABANES,Castelló/Castellón,0.045412,40.156104
132,CABANES,CABANES,Girona,2.977957,42.307504
199,CIEZA,CIEZA,Murcia,-1.427730,38.236594
200,CIEZA,CIEZA,Cantabria,-4.096722,43.221056
340,LA CONCHA,VILLAESCUSA,Zamora,-5.464099,41.206136
341,LA CONCHA,VILLAESCUSA,Cantabria,-3.856639,43.370333
434,MIERES,MIERES,Girona,2.640292,42.123296
435,MIERES,MIERES,Asturias,-5.772657,43.248794
436,MIERES DEL CAMIN,MIERES,Girona,2.640292,42.123296
437,MIERES DEL CAMIN,MIERES,Asturias,-5.772657,43.248794


In [32]:
final_merged.to_csv(r'../BD_MUNICIPIOS-ENTIDADES/DE_MUNICIP_LAT_LONG.csv')